In [1]:
#Importing the necessary libraries 
import jsonlines
import pandas as pd
import numpy as np
import re
from timeit import default_timer as timer
from sklearn.feature_extraction.text import CountVectorizer
from nltk.corpus import stopwords
from nltk.stem import SnowballStemmer
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore')

In [2]:
#Importing the data
newlist = []
with jsonlines.open('categorized-comments.jsonl') as f:
    for obj in f.iter(type = dict, skip_invalid = True):
        newlist.append(obj)

#Turning the list of text dictionaries into a data frame for easy access
comments = pd.DataFrame(newlist)
del newlist

In [3]:
#Balancing the classes by taking 15% of each category
#The 15% rows will be randomly sampled to reduce the size of the dataset
dflist = []
for cat in np.unique(comments['cat']):
    dflist.append(comments[comments['cat'] == cat].sample(frac = 0.15))
comments = pd.concat(dflist)
del dflist

In [4]:
#Cleaning the text by removing punctuation and stopwords and stemming

stemmer = SnowballStemmer('english')
words = stopwords.words('english')

comments['cleaned'] = comments['txt'].apply(lambda x: ' '.join([stemmer.stem(i) for i in re.sub("[^a-zA-Z]", " ", x).split() if i not in words]).lower())
del comments['txt']

In [5]:
#Generating a TFIDF matrix from the cleaned text as input to the scikit neural network model

count = CountVectorizer()
X = count.fit_transform(comments['cleaned']).toarray()

In [6]:
#Splitting the dataset into training and testing sets 
#40% of the data is set aside for testing due to memory limits on training array size

X_train, X_test, y_train, y_test = train_test_split(X, comments['cat'], test_size = 0.4, random_state = 0)
del X

In [7]:
#Turning the categorical target vectors into numerical vectors

y_train = y_train.astype('category').cat.codes
y_test = y_test.astype('category').cat.codes

In [8]:
#Defining a function to build, train, and test the model with 4 layers (2 hidden layers)

def model_build_train_test(X_train, y_train, X_test, hidden_layer_sizes, batch_size, max_iter):
    #Defining the model and fitting it to the training data
    classifier = MLPClassifier(hidden_layer_sizes = hidden_layer_sizes, 
                               batch_size = batch_size, 
                               max_iter = max_iter, 
                               verbose = True, 
                               random_state = 0)
    classifier.fit(X_train, y_train)
    
    #Testing the model on the testing set and returning the predictions
    y_pred = classifier.predict(X_test)
    return y_pred

In [9]:
#Running the model function
y_pred = model_build_train_test(X_train, y_train, X_test, (720, 120), 5000, 5)

Iteration 1, loss = 0.80141044
Iteration 2, loss = 0.51704057
Iteration 3, loss = 0.36472122
Iteration 4, loss = 0.29083128
Iteration 5, loss = 0.24283254


In [12]:
#Displaying the result metrics by comparing the predicted categories to the true categories
#The training accuracies were checked as well, and it was confirmed that overfitting was not occurring

print('Scores')
print('------')
print('Accuracy: {} percent'.format(round(accuracy_score(y_test, y_pred), 3)*100))
print('Precision: {} percent'.format(round(precision_score(y_test, y_pred, average = 'weighted'), 3)*100))
print('Recall: {} percent'.format(round(recall_score(y_test, y_pred, average = 'weighted'), 3)*100))
print('F1 Score: {} percent \n'.format(round(f1_score(y_test, y_pred, average = 'weighted'), 3)*100))
print('Confusion matrix \n----------------\n {}'.format(confusion_matrix(y_test, y_pred)))

Scores
------
Accuracy: 84.3 percent
Precision: 83.8 percent
Recall: 84.3 percent
F1 Score: 83.5 percent 

Confusion matrix 
----------------
 [[  687    72   734]
 [   20  5320  3298]
 [  154  1419 24685]]
